# BreatheEasy Business Agent
## An AI-Powered Customer Service Chatbot

This notebook implements an intelligent chatbot for BreatheEasy, an eco-friendly home cleaning business.

**Created by:** Hassan Khalil  
**Course:** EECE 503P - Assignment 3

## 1. Install Required Dependencies

In [ ]:
# Install required packages
!pip install openai gradio python-dotenv PyPDF2 -q

## 2. Import Libraries

In [ ]:
import os
import json
from datetime import datetime
from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr
from PyPDF2 import PdfReader

print("All libraries imported successfully!")

## 3. Load Environment Variables and API Key

In [ ]:
# Load environment variables from .env file (override any existing ones)
load_dotenv(override=True)

# Get OpenAI API key
api_key = os.getenv('OPENAI_API_KEY')

if not api_key:
    raise ValueError("OpenAI API key not found. Please check your .env file.")

# Initialize OpenAI client
client = OpenAI(api_key=api_key)

print("OpenAI client initialized successfully!")

## 4. Load Business Information

In [ ]:
def load_business_summary():
    """Load business summary from text file"""
    try:
        with open('me/business_summary.txt', 'r', encoding='utf-8') as f:
            return f.read()
    except FileNotFoundError:
        print("Warning: business_summary.txt not found")
        return ""

def load_business_pdf():
    """Load and extract text from business PDF"""
    try:
        reader = PdfReader('me/about_business.pdf')
        text = ""
        for page in reader.pages:
            text += page.extract_text() + "\n"
        return text
    except FileNotFoundError:
        print("Warning: about_business.pdf not found")
        return ""

# Load business information
business_summary = load_business_summary()
business_pdf_content = load_business_pdf()

print("Business information loaded successfully!")
print(f"Summary length: {len(business_summary)} characters")
print(f"PDF content length: {len(business_pdf_content)} characters")

## 5. Define Tool Functions

In [ ]:
# Initialize storage for leads and feedback
customer_leads = []
customer_feedback = []

def record_customer_interest(name, email, message):
    """
    Record customer interest/lead information.
    
    Args:
        name (str): Customer's name
        email (str): Customer's email address
        message (str): Customer's message or interest details
    
    Returns:
        str: Confirmation message
    """
    timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    
    lead_data = {
        'timestamp': timestamp,
        'name': name,
        'email': email,
        'message': message
    }
    
    customer_leads.append(lead_data)
    
    # Log to console
    print("\n" + "="*60)
    print("NEW CUSTOMER LEAD RECORDED")
    print("="*60)
    print(f"Timestamp: {timestamp}")
    print(f"Name: {name}")
    print(f"Email: {email}")
    print(f"Message: {message}")
    print("="*60 + "\n")
    
    # Also save to file
    try:
        with open('customer_leads.json', 'w') as f:
            json.dump(customer_leads, f, indent=2)
    except Exception as e:
        print(f"Error saving lead to file: {e}")
    
    return f"Thank you {name}! Your information has been recorded. We'll contact you at {email} shortly."

def record_feedback(question):
    """
    Record customer feedback or unanswered questions.
    
    Args:
        question (str): The question or feedback that couldn't be answered
    
    Returns:
        str: Confirmation message
    """
    timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    
    feedback_data = {
        'timestamp': timestamp,
        'question': question
    }
    
    customer_feedback.append(feedback_data)
    
    # Log to console
    print("\n" + "="*60)
    print("UNANSWERED QUESTION RECORDED")
    print("="*60)
    print(f"Timestamp: {timestamp}")
    print(f"Question: {question}")
    print("="*60 + "\n")
    
    # Also save to file
    try:
        with open('customer_feedback.json', 'w') as f:
            json.dump(customer_feedback, f, indent=2)
    except Exception as e:
        print(f"Error saving feedback to file: {e}")
    
    return "Your question has been recorded and will be reviewed by our team."

print("Tool functions defined successfully!")

## 6. Define Tool Schemas for OpenAI

In [ ]:
tools = [
    {
        "type": "function",
        "function": {
            "name": "record_customer_interest",
            "description": "Record customer contact information and interest. Use this when a customer wants to schedule a service, request a quote, or leave their contact details for follow-up.",
            "parameters": {
                "type": "object",
                "properties": {
                    "name": {
                        "type": "string",
                        "description": "The customer's full name"
                    },
                    "email": {
                        "type": "string",
                        "description": "The customer's email address"
                    },
                    "message": {
                        "type": "string",
                        "description": "Details about their interest, service needed, or any specific requirements"
                    }
                },
                "required": ["name", "email", "message"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "record_feedback",
            "description": "Record customer questions or feedback that you cannot answer. Use this when you don't know the answer to a question or when the customer asks about something not covered in your knowledge base.",
            "parameters": {
                "type": "object",
                "properties": {
                    "question": {
                        "type": "string",
                        "description": "The question or feedback that could not be answered"
                    }
                },
                "required": ["question"]
            }
        }
    }
]

print("Tool schemas defined successfully!")

## 7. Create System Prompt

In [ ]:
system_prompt = f"""
You are a friendly and knowledgeable customer service representative for BreatheEasy, an eco-friendly home cleaning business.

BUSINESS CONTEXT:
{business_summary}

YOUR ROLE:
- Answer questions about BreatheEasy's services, pricing, team, and values
- Help customers understand our unique allergy-safe and eco-friendly approach
- Collect customer contact information when they express interest in our services
- Be warm, professional, and health-conscious in your responses
- Emphasize our commitment to health, safety, and environmental responsibility

IMPORTANT GUIDELINES:
1. When customers ask about scheduling, pricing, or want to book a service, ask for their name, email, and service details, then use the record_customer_interest function.
2. If you don't know the answer to a question, be honest and use the record_feedback function to log it for follow-up.
3. Encourage customers to leave their contact information so we can provide personalized service.
4. Highlight our unique value propositions: allergy-safe protocol, product transparency, and health-first approach.
5. Be conversational and empathetic, especially when customers mention allergies or health concerns.
6. Keep responses concise but informative.

SERVICES WE OFFER:
1. Deep Cleaning Services - Comprehensive home cleaning with allergen elimination
2. Move-In/Move-Out Cleaning - Thorough preparation for new occupants
3. Allergen Treatment Services - Specialized for allergy sufferers
4. Regular Maintenance Cleaning - Weekly, bi-weekly, or monthly schedules

CONTACT INFO:
- Email: hello@breatheeasy.com
- Phone: (555) 123-EASY
- Hours: Monday-Saturday, 8am-6pm

Remember: You represent a business that genuinely cares about customer health and the environment. Let that shine through in every interaction!
"""

print("System prompt created successfully!")

## 8. Implement Chat Function with Tool Calling

In [ ]:
def chat_with_breatheeasy(message, history):
    """
    Main chat function that handles user messages and tool calls.
    
    Args:
        message (str): User's message
        history (list): Chat history in Gradio format
    
    Returns:
        str: Assistant's response
    """
    # Convert Gradio history format to OpenAI format
    messages = [{"role": "system", "content": system_prompt}]
    
    # Add conversation history
    for human, assistant in history:
        messages.append({"role": "user", "content": human})
        messages.append({"role": "assistant", "content": assistant})
    
    # Add current message
    messages.append({"role": "user", "content": message})
    
    # Call OpenAI API with tools
    try:
        response = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=messages,
            tools=tools,
            tool_choice="auto"
        )
        
        response_message = response.choices[0].message
        
        # Check if the model wants to call a tool
        if response_message.tool_calls:
            # Process tool calls
            messages.append(response_message)
            
            for tool_call in response_message.tool_calls:
                function_name = tool_call.function.name
                function_args = json.loads(tool_call.function.arguments)
                
                # Execute the appropriate function
                if function_name == "record_customer_interest":
                    function_response = record_customer_interest(
                        name=function_args.get("name"),
                        email=function_args.get("email"),
                        message=function_args.get("message")
                    )
                elif function_name == "record_feedback":
                    function_response = record_feedback(
                        question=function_args.get("question")
                    )
                else:
                    function_response = "Unknown function called."
                
                # Add function response to messages
                messages.append({
                    "tool_call_id": tool_call.id,
                    "role": "tool",
                    "name": function_name,
                    "content": function_response,
                })
            
            # Get final response from the model
            second_response = client.chat.completions.create(
                model="gpt-4o-mini",
                messages=messages
            )
            
            return second_response.choices[0].message.content
        
        else:
            # No tool call, return the regular response
            return response_message.content
            
    except Exception as e:
        error_msg = f"I apologize, but I encountered an error: {str(e)}. Please try again or contact us directly at hello@breatheeasy.com"
        print(f"Error in chat function: {e}")
        return error_msg

print("Chat function implemented successfully!")

## 9. Create Gradio Interface

In [ ]:
# Create custom CSS for branding
custom_css = """
.gradio-container {
    font-family: 'Arial', sans-serif;
}
#component-0 {
    background: linear-gradient(135deg, #667eea 0%, #764ba2 100%);
    color: white;
}
"""

# Create the Gradio ChatInterface
demo = gr.ChatInterface(
    fn=chat_with_breatheeasy,
    title="🌿 BreatheEasy - Eco-Friendly Home Cleaning",
    description="Welcome! I'm here to help you learn about our allergy-safe, eco-friendly cleaning services. Ask me anything about our services, pricing, or schedule a cleaning!",
    examples=[
        "What services do you offer?",
        "Tell me about your allergy-safe cleaning protocol",
        "I'm interested in scheduling a deep cleaning",
        "What makes BreatheEasy different from other cleaning services?",
        "Do you use eco-friendly products?",
        "I have severe allergies. Can you help?"
    ],
    theme=gr.themes.Soft()
)

print("Gradio interface created successfully!")

## 10. Launch the Chatbot

In [ ]:
# Launch the interface
if __name__ == "__main__":
    demo.launch(
        share=True,  # Creates a public link
        debug=True,
        show_error=True
    )

## 11. View Collected Data (Optional)

In [ ]:
# View collected customer leads
print("\n" + "="*60)
print("CUSTOMER LEADS COLLECTED")
print("="*60)
if customer_leads:
    for i, lead in enumerate(customer_leads, 1):
        print(f"\nLead #{i}:")
        print(f"  Timestamp: {lead['timestamp']}")
        print(f"  Name: {lead['name']}")
        print(f"  Email: {lead['email']}")
        print(f"  Message: {lead['message']}")
else:
    print("No leads collected yet.")
print("="*60)

# View unanswered questions/feedback
print("\n" + "="*60)
print("CUSTOMER FEEDBACK / UNANSWERED QUESTIONS")
print("="*60)
if customer_feedback:
    for i, feedback in enumerate(customer_feedback, 1):
        print(f"\nFeedback #{i}:")
        print(f"  Timestamp: {feedback['timestamp']}")
        print(f"  Question: {feedback['question']}")
else:
    print("No feedback recorded yet.")
print("="*60)

## Testing the Chatbot

Try these test scenarios:

1. **Service Inquiry**: "What services do you offer?"
2. **Allergy Concern**: "I have severe allergies. Can you help?"
3. **Lead Collection**: "I'd like to schedule a deep cleaning for next week"
4. **Product Transparency**: "What cleaning products do you use?"
5. **Pricing Question**: "How much does a regular maintenance cleaning cost?"
6. **Unknown Question**: "Do you offer carpet installation?" (should trigger feedback recording)

The chatbot should:
- Answer questions about BreatheEasy using the business information
- Ask for contact details when customers express interest
- Use the `record_customer_interest` tool to save leads
- Use the `record_feedback` tool for questions it cannot answer
- Maintain a friendly, professional, health-conscious tone